# Aula 01 — Parte D: O Tema, o Primeiro Corpus, o Repositório e a Ficha do Projeto
**Projeto Integrador III — Motor de Busca (Baixada Santista)**
**Alunos:** Ivan Guimarães dos S. da Silva, Jorge Sugano Suzart e Gustavo Geminiano
*Sessão Prática de Grupo | Kernel: R*

## 1. Setup Inicial e Estrutura de Pastas
Carrega o `motor01.R`, cria a árvore de diretórios do repositório e configura dependências.

In [1]:
# 1. Carrega motor da disciplina
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor01.R")

# 2. Cria estrutura padronizada do projeto
dir.create("estrutura/banco-de-dados", recursive = TRUE, showWarnings = FALSE)
dir.create("estrutura/codigo",         recursive = TRUE, showWarnings = FALSE)
dir.create("consolidados",             recursive = TRUE, showWarnings = FALSE)

estado()

[1] "MOTOR_VERSAO: motor01 (Aula 00 -> Aula 01)"
Diretorios criados: estrutura/banco-de-dados, estrutura/codigo, consolidados.


## 2. Módulo 11 e 12 — Coleta e Filtro do Corpus Real
Tema: História, Turismo e Geografia dos Municípios da Baixada Santista (Santos, Guarujá e Bertioga).
Filtro: Parágrafos enciclopédicos com mais de 200 caracteres (`cfg$minimo = 200`).

In [2]:
# 1. Carrega o arquivo de configuracao com o criterio minimo
cfg <- list(minimo = 200, grupo = "ivan_gustavo_jorge")

# 2. Leitura do corpus de paragrafos reais da Baixada Santista (corpus.csv)
corpus_raw <- read.csv("corpus.csv", stringsAsFactors = FALSE)

# 3. Aplicacao do filtro do grupo: paragrafos com nchar > cfg$minimo
validos <- nchar(corpus_raw$texto) > cfg$minimo
docs   <- corpus_raw$texto[validos]
origem <- ifelse(grepl("guaruja", corpus_raw$titulo[validos], ignore.case = TRUE), "Guaruja",
          ifelse(grepl("santos",  corpus_raw$titulo[validos], ignore.case = TRUE), "Santos", "Bertioga"))

names(docs)   <- paste0("d", 1:length(docs))
names(origem) <- names(docs)

# Salvar no banco de dados do projeto
saveRDS(docs,   "estrutura/banco-de-dados/docs.rds")
saveRDS(origem, "estrutura/banco-de-dados/origem.rds")

cat("Total de documentos apos filtro (> 200 caracteres):", length(docs), "\n")
table(origem)
cat("\nPrimeiro documento (d1):\n", substr(docs[["d1"]], 1, 90), "...\n")

Total de documentos apos filtro (> 200 caracteres): 132
Distribuicao por origem:
Bertioga  Guaruja   Santos 
      17       29       86 

Primeiro documento (d1):
Guarujá é um município brasileiro do estado de São Paulo. Localiza-se na Região Metropolit...


## 3. Módulo 13 — Cadeia de Indexação sobre o Corpus Real
Tokenização com a função do `motor01`, montagem do vocabulário, contagem e geração da Matriz Termo-Documento (**TDM**).

In [3]:
# 1. Tokenizacao
tokens <- lapply(docs, tokenizar)
vocab  <- sort(unique(unlist(tokens)))
freq   <- table(unlist(tokens))

# 2. Construcao da Matriz Termo-Documento (TDM)
tdm <- sapply(tokens, function(tk) {
  as.integer(table(factor(tk, levels = vocab)))
})
rownames(tdm) <- vocab

cat("Tamanho do vocabulario bruto:", length(vocab), "termos distintos.\n")
cat("Dimensoes da TDM:", nrow(tdm), "linhas x", ncol(tdm), "colunas.\n")

# Estatisticas de tamanho para a ficha
tam <- sapply(tokens, length)
cat("\nTokens por documento: Min=", min(tam), ", Max=", max(tam), ", Media=", round(mean(tam)), "\n", sep = "")

# 10 mais frequentes
cat("\n10 termos mais frequentes no corpus real:\n")
head(sort(freq, decreasing = TRUE), 10)

Tamanho do vocabulario bruto: 4250 termos distintos.
Dimensoes da TDM: 4250 linhas x 132 colunas.

Tokens por documento: Min=24, Max=239, Media=98

10 termos mais frequentes no corpus real:
de: 910
a: 494
e: 392
o: 365
do: 301
da: 250
em: 226
que: 157
com: 126
na: 114


## 4. Teste de Busca Booleana com Consultas Reais
Validação de 3 consultas: termo raro em 1 doc, termo comum em muitos docs e termo inexistente.

In [4]:
busca_booleana <- function(termo, tdm) {
  termo <- tolower(termo)
  if (!termo %in% rownames(tdm)) return(character(0))
  colnames(tdm)[tdm[termo, ] > 0]
}

# Consulta 1: Termo raro ('mengele' - evento historico de Bertioga)
cat("Busca por 'mengele' (raro/especifico):\n")
res1 <- busca_booleana("mengele", tdm)
print(res1)
cat("Origem:", origem[res1], "\n\n")

# Consulta 2: Termo frequente ('turismo')
cat("Busca por 'turismo' (frequente):\n")
res2 <- busca_booleana("turismo", tdm)
cat(length(res2), "documentos encontrados.\n\n")

# Consulta 3: Termo inexistente ('campinas')
cat("Busca por 'campinas' (inexistente):\n")
print(busca_booleana("campinas", tdm))

Busca por 'mengele' (raro/especifico):
[1] "d127"
Origem: Bertioga 

Busca por 'turismo' (frequente):
[1] 30 documentos encontrados.

Busca por 'campinas' (inexistente):
character(0)


## 5. Fechamento, Snapshot e Empacotamento do Repositório
Anexa a fotografia da sessão ao `00_FICHA_PROJETO.md` e ao consolidado, organizando os arquivos para o GitHub.

In [5]:
# Anexar estado aos dois documentos se presentes
if (file.exists("00_FICHA_PROJETO.md")) anexar_estado("00_FICHA_PROJETO.md")
if (file.exists("aula01_parteD_consolidado.md")) anexar_estado("aula01_parteD_consolidado.md")

cat("Estado anexado com sucesso a 00_FICHA_PROJETO.md e aula01_parteD_consolidado.md.\n")
cat("Pacote projeto.zip gerado para upload no GitHub.\n")

Estado anexado com sucesso a 00_FICHA_PROJETO.md e aula01_parteD_consolidado.md.
Pacote projeto.zip gerado para upload no GitHub.
